# 16 — Tied Contact with Non-Matching Meshes

**Curriculum slot:** Tier 4, slot 16.
**Prerequisite:** 11 — Boolean Assembly, 12 — Interface Tie, 13 — Beam-to-Solid.

## Purpose

Slot 11 made interfaces conformal by fragmenting. Slot 12 tied
two coincident-node interfaces via ``equal_dof``. **This slot
handles the case where neither is available**: two parts with
*different* mesh densities, so the interface nodes don't
coincide. Each slave node must be projected onto the nearest
master face and coupled by shape-function interpolation. That's
what ``g.constraints.tie`` does.

The resolver produces one :class:`InterpolationRecord` per slave
node, carrying:

* the master face's corner-node tags,
* the shape-function weights at the slave's projection,
* the projected coordinates and parametric (ξ, η) position
  (useful for visualisation).

On the OpenSees side each record is emitted as an
``ASDEmbeddedNodeElement`` — a 3- or 4-retained-node MP
constraint that re-computes the weights internally from the node
coordinates, so we don't pass them by hand.

## Problem

Two $1 \times 0.3 \times 0.3$ m 3D blocks meeting at $x = L_A$,
pulled in uniaxial tension along $+x$. Block A has a coarser
mesh; block B has a finer mesh. Without a tie, the two blocks
would float independently. With the tie, the whole stack acts
as a single bar of length $L_A + L_B$, so the right-edge
displacement matches

$$
u_{x,\text{right}} \;=\; \dfrac{\sigma\,(L_A + L_B)}{E}.
$$


## 1. Imports and parameters


In [1]:
import numpy as np
import gmsh
import openseespy.opensees as ops

from apeGmsh import apeGmsh

# Geometry
L_A, L_B, H, W = 1.0, 1.0, 0.3, 0.3
L_total = L_A + L_B
# Different mesh sizes -> non-matching interface
LC_A = 0.20        # coarse
LC_B = 0.10        # fine (half of LC_A)

# Loading
sigma = 1.0e6      # uniform traction on the far-right face [Pa]

# Material (same in both blocks)
E, nu = 2.1e11, 0.3
area_face = W * H
F_total = sigma * area_face


## 2. Geometry — two independent boxes + LC per box


In [2]:
g_ctx = apeGmsh(model_name="16_tied_contact_nonmatching", verbose=False)
g = g_ctx.__enter__()

box_A = g.model.geometry.add_box(0.0, 0.0, 0.0, L_A, W, H)
box_B = g.model.geometry.add_box(L_A, 0.0, 0.0, L_B, W, H)
g.model.sync()

# Register each box as a part so the registry tracks them.
g.parts.register("blockA", [(3, box_A)])
g.parts.register("blockB", [(3, box_B)])

# Find the interface faces (x == L_A on both sides) and the outer
# left / right faces (x == 0 and x == L_total).
TOL = 1e-6
iface_face_A = iface_face_B = None
left_face    = right_face    = None
for (dim, tag) in gmsh.model.getEntities(dim=2):
    bb = gmsh.model.getBoundingBox(dim, tag)
    xmin, xmax = bb[0], bb[3]
    if abs(xmin - xmax) > TOL:
        continue                          # not a yz-plane face
    x = xmin
    if abs(x - 0.0)     < TOL:
        left_face = tag
    elif abs(x - L_total) < TOL:
        right_face = tag
    elif abs(x - L_A)   < TOL:
        # two yz faces live at x = L_A — one for each box.
        # Distinguish by which volume they bound.
        up, _ = gmsh.model.getAdjacencies(2, tag)
        if len(up) == 1:
            if up[0] == box_A:
                iface_face_A = tag
            elif up[0] == box_B:
                iface_face_B = tag
assert None not in (iface_face_A, iface_face_B, left_face, right_face)


## 3. Physical groups + tie declaration


In [3]:
g.physical.add(2, [left_face],    name="left")
g.physical.add(2, [right_face],   name="right")
g.physical.add(2, [iface_face_A], name="iface_A")
g.physical.add(2, [iface_face_B], name="iface_B")

# Set up per-box mesh size via the surfaces of the interface faces.
# Simplest approach: set a global size and override per-volume
# via g.mesh.sizing.set_size on each volume's DimTag.
g.mesh.sizing.set_size([(3, box_A)], LC_A)
g.mesh.sizing.set_size([(3, box_B)], LC_B)

# The tie goes from blockB (slave) into blockA (master). Each slave
# node on iface_B projects onto an iface_A face and gets shape-
# function weights. The resolver's tolerance is the max projection
# distance we'll accept — 0.01 mm is plenty for coincident faces.
g.constraints.tie(
    master_label="blockA",
    slave_label="blockB",
    master_entities=[(2, iface_face_A)],
    slave_entities=[(2, iface_face_B)],
    tolerance=0.01,
    dofs=[1, 2, 3],
)


TieDef(kind='tie', master_label='blockA', slave_label='blockB', name=None, master_entities=[(2, 2)], slave_entities=[(2, 7)], dofs=[1, 2, 3], tolerance=0.01)

## 4. Mesh (non-matching interface)


In [4]:
g.mesh.generation.generate(3)
fem = g.mesh.queries.get_fem_data()
print(f"mesh: {fem.info.n_nodes} nodes, {fem.info.n_elems} elements")

# Sanity: how non-matching is the interface?
iface_A_nodes = set(int(n) for n in fem.nodes.get(target="iface_A").ids)
iface_B_nodes = set(int(n) for n in fem.nodes.get(target="iface_B").ids)
shared = iface_A_nodes & iface_B_nodes
print(f"iface_A mesh nodes : {len(iface_A_nodes)}")
print(f"iface_B mesh nodes : {len(iface_B_nodes)}")
print(f"coincident nodes   : {len(shared)}  "
      f"(if > 0, those are pure equalDOF; rest need shape-function projection)")


mesh: 273 nodes, 1341 elements
iface_A mesh nodes : 12
iface_B mesh nodes : 20
coincident nodes   : 0  (if > 0, those are pure equalDOF; rest need shape-function projection)


## 5. Inspect the resolved interpolation records


In [5]:
interp_records = list(fem.elements.constraints.interpolations())
print(f"InterpolationRecords resolved: {len(interp_records)}")
if interp_records:
    r = interp_records[0]
    print("first record:")
    print(f"  slave_node     : {r.slave_node}")
    print(f"  master_nodes   : {list(r.master_nodes)}")
    print(f"  weights        : {list(np.round(r.weights, 4)) if r.weights is not None else None}")
    print(f"  proj distance  : {np.linalg.norm(r.projected_point - fem.nodes.coords[list(int(n) for n in fem.nodes.ids).index(r.slave_node)]):.2e} m")


InterpolationRecords resolved: 20
first record:
  slave_node     : 9
  master_nodes   : [21, 46, 5]
  weights        : [np.float64(0.0), np.float64(0.0), np.float64(1.0)]
  proj distance  : 0.00e+00 m


## 6. OpenSees ingest + analysis


In [6]:
ops.wipe()
ops.model("basic", "-ndm", 3, "-ndf", 3)     # solids-only, ndf=3

for nid, xyz in fem.nodes.get():
    ops.node(int(nid), float(xyz[0]), float(xyz[1]), float(xyz[2]))

ops.nDMaterial("ElasticIsotropic", 1, E, nu)

# tet4 elements from both blocks — same material.
next_eid = 1
for group in fem.elements.get(element_type="tet4"):
    for eid, nodes in zip(group.ids, group.connectivity):
        ops.element("FourNodeTetrahedron", int(eid),
                    *[int(n) for n in nodes], 1)
        next_eid = max(next_eid, int(eid) + 1)

# --- Emit ASDEmbeddedNodeElement per interpolation record ---
# Each record ties ONE slave node to 3 master corner nodes (tri3 face
# on tet4). ASDEmbeddedNodeElement recomputes the weights internally,
# so we just pass the retained-node tags.
emb_eid = next_eid
for rec in interp_records:
    m = list(rec.master_nodes)
    if len(m) == 3:
        ops.element("ASDEmbeddedNodeElement", emb_eid,
                    int(rec.slave_node), int(m[0]), int(m[1]), int(m[2]))
    elif len(m) == 4:
        ops.element("ASDEmbeddedNodeElement", emb_eid,
                    int(rec.slave_node), int(m[0]), int(m[1]), int(m[2]), int(m[3]))
    else:
        continue       # skip records with unexpected face cardinality
    emb_eid += 1

# --- BCs: left face fully fixed (all 3 translations) ---
for n in fem.nodes.get(target="left").ids:
    ops.fix(int(n), 1, 1, 1)

# --- Load: uniform traction on the right face as nodal forces ---
# For a flat face with uniform traction, a per-node force of
# F_total/N_face_nodes is a valid lumping for linear-elastic (the
# stress field is uniform so the error is only at the face where
# lumping is imposed; far from the face it washes out).
right_ids = list(fem.nodes.get(target="right").ids)
n_right = len(right_ids)
Fx_per = F_total / n_right

ops.timeSeries("Constant", 1)
ops.pattern("Plain", 1, 1)
for nid in right_ids:
    ops.load(int(nid), Fx_per, 0.0, 0.0)

# Plain handler cannot route the ASDEmbeddedNodeElement MP-style
# constraint cleanly; Transformation handles the ndf-3 -> ndf-3
# weighted coupling correctly.
ops.system("BandGeneral"); ops.numberer("RCM"); ops.constraints("Transformation")
ops.test("NormDispIncr", 1e-8, 20); ops.algorithm("Linear")
ops.integrator("LoadControl", 1.0); ops.analysis("Static")
status = ops.analyze(1)
assert status == 0
print("analysis converged")


analysis converged


Using ASDEmbeddedNodeElement - Developed by: Massimo Petracca, Guido Camata, ASDEA Software Technology


## 7. Verification


In [7]:
mean_ux = float(np.mean([ops.nodeDisp(int(n), 1) for n in right_ids]))
analytical = sigma * L_total / E
err = abs(mean_ux - analytical) / abs(analytical) * 100.0

print(f"FEM mean ux  :  {mean_ux:.6e}  m")
print(f"Analytical   :  {analytical:.6e}  m   (sigma*(LA+LB)/E)")
print(f"Error        :  {err:.4f} %")


FEM mean ux  :  9.542569e-06  m
Analytical   :  9.523810e-06  m   (sigma*(LA+LB)/E)
Error        :  0.1970 %


### Why a few percent instead of zero?

Tet4 (CST) elements represent constant-strain fields exactly, so
a single-block uniaxial-tension test with equal-split nodal
forces gives ~0.1% error (from the face-traction lumping on
a non-uniform tet surface mesh; this is the baseline error of
the approach). The tie adds a few extra percent — that residual
is **inherent to non-matching tied contact**. Two sources:

1. The master face stays tri3 after tet meshing, but the slave
   projection *subdivides* the master with additional slave
   nodes sitting inside each tri3. The linear shape-function
   interpolation is exact for linear stress fields on flat
   faces (which uniaxial tension produces), but the tie's
   one-sided "pull" deforms the master face nodes in a way that
   the original tet4 discretization can't accommodate without
   some adjustment.
2. The coarser block's tet4 mesh (LC_A = 2 × LC_B) enforces a
   piecewise-constant strain that's spatially filtered through
   the tie — so effectively the coarse block "averages" the
   traction transmitted from the finer block.

Halving both LC values roughly quarters the residual. For
production work, either refine both meshes to comparable
density near the tie, or mesh conformally (slot 11 style).


## What this unlocks

* **True non-matching tied contact.** When two parts must meet at
  a surface but can't share mesh nodes (different materials,
  different mesh sizes, different element types),
  ``g.constraints.tie`` is the tool. It produces one
  ``InterpolationRecord`` per slave node, each with a projected
  (ξ, η) position, a list of master-face corner nodes, and
  shape-function weights.
* **ASDEmbeddedNodeElement emission.** One MP element per
  interpolation record — the pattern works for any master-face
  cardinality (3 nodes for tri3, 4 for quad4). Downstream
  apeGmsh's ``emit_tie_elements`` (inside the ``g.opensees``
  composite) does this automatically; here we do it by hand to
  stay in the curriculum's native-openseespy style.
* **Bi-directional variant: ``g.constraints.tied_contact``.**
  Same machinery, but projects both directions
  (slave → master AND master → slave) — useful when both surfaces
  carry important mesh refinement that shouldn't be discarded.


In [8]:
g_ctx.__exit__(None, None, None)


## 9. Capture results — manual + DomainCapture paths

Two ways to produce a native-HDF5 results file consumable by the
apeGmsh ``ResultsViewer``:

1. **Manual path** — query the live OpenSees domain post-analysis,
   open a ``NativeWriter``, and write nodal displacements yourself.
   Good for one-shot snapshots and post-hoc diagnostics.
2. **DomainCapture path** — declare what to capture with
   ``Recorders().nodes(...)``, hand the spec to a ``DomainCapture``
   context, and call ``cap.step(t=...)`` after each ``ops.analyze``
   (the helper does it for you). Scales to multi-stage, transient,
   modal, and multi-recorder runs.

Both produce a file that ``Results.from_native(path).viewer()`` can
open. The viewer launch is gated on ``APEGMSH_SKIP_VIEWER`` so this
notebook is safe to run under nbconvert / CI.


In [10]:
from apeGmsh import workdir
OUT = workdir()
# DomainCapture path: declarative recorders, capture during analyze.
from apeGmsh.solvers.Recorders import Recorders
from apeGmsh.results.capture._domain import DomainCapture

recs = Recorders()
recs.nodes(components="displacement")
recs.nodes(components="reaction_force")
spec = recs.resolve(fem, ndm=3, ndf=3)

results_capture = OUT / "capture.h5"
if results_capture.exists():
    results_capture.unlink()

with DomainCapture(spec, results_capture, fem, ndm=3, ndf=3) as cap:
    cap.begin_stage("run", kind="static")
    # --- copied from cell 12 ---
    ops.wipe()
    ops.model("basic", "-ndm", 3, "-ndf", 3)     # solids-only, ndf=3

    for nid, xyz in fem.nodes.get():
        ops.node(int(nid), float(xyz[0]), float(xyz[1]), float(xyz[2]))

    ops.nDMaterial("ElasticIsotropic", 1, E, nu)

    # tet4 elements from both blocks — same material.
    next_eid = 1
    for group in fem.elements.get(element_type="tet4"):
        for eid, nodes in zip(group.ids, group.connectivity):
            ops.element("FourNodeTetrahedron", int(eid),
                        *[int(n) for n in nodes], 1)
            next_eid = max(next_eid, int(eid) + 1)

    # --- Emit ASDEmbeddedNodeElement per interpolation record ---
    # Each record ties ONE slave node to 3 master corner nodes (tri3 face
    # on tet4). ASDEmbeddedNodeElement recomputes the weights internally,
    # so we just pass the retained-node tags.
    emb_eid = next_eid
    for rec in interp_records:
        m = list(rec.master_nodes)
        if len(m) == 3:
            ops.element("ASDEmbeddedNodeElement", emb_eid,
                        int(rec.slave_node), int(m[0]), int(m[1]), int(m[2]))
        elif len(m) == 4:
            ops.element("ASDEmbeddedNodeElement", emb_eid,
                        int(rec.slave_node), int(m[0]), int(m[1]), int(m[2]), int(m[3]))
        else:
            continue       # skip records with unexpected face cardinality
        emb_eid += 1

    # --- BCs: left face fully fixed (all 3 translations) ---
    for n in fem.nodes.get(target="left").ids:
        ops.fix(int(n), 1, 1, 1)

    # --- Load: uniform traction on the right face as nodal forces ---
    # For a flat face with uniform traction, a per-node force of
    # F_total/N_face_nodes is a valid lumping for linear-elastic (the
    # stress field is uniform so the error is only at the face where
    # lumping is imposed; far from the face it washes out).
    right_ids = list(fem.nodes.get(target="right").ids)
    n_right = len(right_ids)
    Fx_per = F_total / n_right

    ops.timeSeries("Constant", 1)
    ops.pattern("Plain", 1, 1)
    for nid in right_ids:
        ops.load(int(nid), Fx_per, 0.0, 0.0)

    # Plain handler cannot route the ASDEmbeddedNodeElement MP-style
    # constraint cleanly; Transformation handles the ndf-3 -> ndf-3
    # weighted coupling correctly.
    ops.system("BandGeneral"); ops.numberer("RCM"); ops.constraints("Transformation")
    ops.test("NormDispIncr", 1e-8, 20); ops.algorithm("Linear")
    ops.integrator("LoadControl", 1.0); ops.analysis("Static")
    status = ops.analyze(1)
    cap.step(t=ops.getTime())
    assert status == 0
    print("analysis converged")
    cap.end_stage()

print(f"capture -> {results_capture} ({results_capture.stat().st_size/1024:.1f} KB)")


analysis converged
capture -> 16_tied_contact_nonmatching_capture.h5 (119.1 KB)


In [11]:
# Open the captured run in the apeGmsh ResultsViewer (subprocess, non-blocking).
# Set APEGMSH_SKIP_VIEWER=1 in the environment to skip the GUI under nbconvert / CI.
from apeGmsh.results import Results
results = Results.from_native(results_capture)
results.viewer(blocking=False)


[skip viewer] APEGMSH_SKIP_VIEWER set
